Most of this is already writting in the labelling_data file. 



In [1]:
from rdkit.Chem.PandasTools import LoadSDF
from rdkit import RDLogger

RDLogger.DisableLog('rdApp.warning')   

# Load the unlabelled dataset
data_warrior_3D_file = '../datasets/druglike_b3db.sdf'

df = LoadSDF(data_warrior_3D_file, smilesName='SMILES')
SMILES = df['SMILES']
supervised_label_df = df[['SMILES', 'logBB', 'Class']]


In [ ]:
from rdkit.Chem.PandasTools import LoadSDF
from rdkit import RDLogger
import pandas as pd
from rdkit import Chem 
from rdkit.Chem import AllChem, MACCSkeys 
import numpy as np

RDLogger.DisableLog('rdApp.warning')   

# ECFP
fpgen = AllChem.GetMorganGenerator(radius=2)
ecfp_fingerprints = []
for smile in SMILES:
    mol = Chem.MolFromSmiles(smile)
    fp = Chem.rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, 2).ToBitString()
    ecfp_fingerprints.append(list(fp))

ecfp_col_names = [f"ECFP_{i}" for i in range(0,2048)]
ecfp_df = pd.DataFrame(ecfp_fingerprints, columns=ecfp_col_names)

# Mordred
mordred_output_file = "../datasets/b3db_drug_like_3d_mordred_labels.csv"

! python -m mordred -3 {data_warrior_3D_file} -o {mordred_output_file}
mordred_df = pd.read_csv(mordred_output_file)


# MACCS
def get_maccs_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        return list(MACCSkeys.GenMACCSKeys(mol).ToList())

col_names = [f"MACCS_{i}" for i in range(0,167)]
maccs_fps = []
for smile in SMILES: 
    maccs_fps.append(get_maccs_fingerprint(smile))

maccs_fp_df = pd.DataFrame(data=maccs_fps, columns=col_names)



# Save dataset:
all_labelled_df = pd.concat([supervised_label_df, ecfp_df, mordred_df.iloc[:, 1:], maccs_fp_df], axis=1)
inverted_classes = (1-np.array([int(i)for i in np.array(all_labelled_df['Class'])]))*0.5
all_labelled_df['Class'] = [int(ilabel) for ilabel in inverted_classes]
all_labelled_df.to_csv('../datasets/druglike_b3db_labelled.csv')

[15:41:06] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[15:41:06] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[15:41:06] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[15:41:06] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[15:41:06] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[15:41:06] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[15:41:06] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[15:41:06] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[15:41:06] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol